In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch                                                     # PyTorch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor  # the VLM and its input preparer

model_id = "Qwen/Qwen2-VL-2B-Instruct"                           # small vision-language model (~2 billion weights)
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda")      # load on the GPU in half precision (fits the T4)
vlm_processor = AutoProcessor.from_pretrained(
    model_id, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)  # resizes photos to at most ~1 million pixels
print("loaded on", vlm.device)                                   # should say cuda

In [ ]:
import requests, json, re                                        # internet, JSON reading, text search
from io import BytesIO                                           # bytes -> file in memory
from PIL import Image                                            # pictures

headers = {"User-Agent": "FoodScanner student project (university coursework)"}  # tell OFF who we are
api = "https://world.openfoodfacts.org/api/v2/product/44000053871?fields=image_nutrition_url"  # lemon creme cookies
nutr_url = requests.get(api, headers=headers, timeout=10).json()["product"]["image_nutrition_url"]  # its nutrition photo link
img = Image.open(BytesIO(requests.get(nutr_url.replace(".400.", ".full."), headers=headers, timeout=20).content)).convert("RGB")  # full-size photo

prompt = ("Read the nutrition information panel in this photo. "
          "Return ONLY a JSON object with these keys: protein_g, fat_g, carb_g, sugars_g, fiber_g, "
          "salt_g, sodium_mg, basis, serving_size_g. "
          "If there is a per 100 g (or per 100 ml) column, use it and set basis to \"per100\"; "
          "otherwise use the per serving values and set basis to \"per_serving\". "
          "Copy numbers exactly as printed. Use null for anything not shown.")  # our instruction ("the code")

messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]  # chat: one photo + the instruction
chat = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)   # format it the way Qwen expects
inputs = vlm_processor(text=[chat], images=[img], return_tensors="pt").to("cuda")               # photo + text -> model input on GPU

with torch.no_grad():                                            # no training
    out = vlm.generate(**inputs, max_new_tokens=200, do_sample=False)  # write the answer; do_sample=False = same answer every run
answer = vlm_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]  # keep only the new text
print(answer)                                                    # the model's reply

In [ ]:
prompt = ("Read the nutrition information panel in this photo. "
          "Return ONLY a JSON object with these keys: protein_g, fat_g, carb_g, sugars_g, fiber_g, "
          "salt_g, sodium_amount, sodium_unit, basis, serving_size_g. "
          "salt_g: ONLY if the panel has a row called Salt, otherwise null. "
          "sodium_amount and sodium_unit (\"mg\" or \"g\"): ONLY if the panel has a row called Sodium, otherwise null. "
          "If there is a per 100 g (or per 100 ml) column, use it and set basis to \"per100\"; "
          "otherwise use the per serving values and set basis to \"per_serving\". "
          "Copy numbers exactly as printed, without doing any calculation. Use null for anything not shown.")  # sharper instruction

def ask_vlm(img):                                                # photo -> the model's reply as a dictionary
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]  # photo + instruction
    chat = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)  # Qwen's chat format
    inputs = vlm_processor(text=[chat], images=[img], return_tensors="pt").to("cuda")  # -> model input on GPU
    with torch.no_grad():                                        # no training
        out = vlm.generate(**inputs, max_new_tokens=200, do_sample=False)  # write the answer (same every run)
    answer = vlm_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]  # new text only
    m = re.search(r"\{.*\}", answer, re.DOTALL)                  # find the {...} part (models sometimes add extra words)
    try:                                                         # try to read it as JSON
        return json.loads(m.group(0)) if m else {}               # text -> dictionary
    except Exception:                                            # broken JSON
        return {}                                                # treat as nothing read

def num(x):                                                      # "less than 1g", "<0.5", "29g", 7 -> a plain number (or None)
    if x is None:                                                # nothing there
        return None
    m = re.search(r"\d+(?:\.\d+)?", str(x).replace(",", "."))    # first number in it (decimal comma -> point)
    return float(m.group(0)) if m else None                      # number, or None if there isn't one

def to_per100(d):                                                # model's reply -> nutrients per 100 g (OUR code does the maths)
    if not d:                                                    # empty reply
        return {}
    factor = 1.0                                                 # per 100 g already
    if d.get("basis") == "per_serving":                          # per serving label
        s = num(d.get("serving_size_g"))                         # serving size in g
        if not s:                                                # missing or 0
            return {}                                            # can't convert
        factor = 100 / s                                         # per serving -> per 100 g
    out = {}                                                     # results
    for key in ["protein_g", "fat_g", "carb_g", "sugars_g", "fiber_g"]:  # the five gram nutrients
        v = num(d.get(key))                                      # its number
        if v is not None and v * factor <= 100:                  # present and plausible
            out[key] = round(v * factor, 3)                      # store per 100 g
    sod = num(d.get("sodium_amount"))                            # sodium, if the label has it
    if sod is not None:                                          # US-style
        unit = str(d.get("sodium_unit") or "mg").lower()         # its unit (assume mg if missing)
        out["sodium_g"] = round((sod / 1000 if unit.startswith("mg") else sod) * factor, 3)  # mg -> g, per 100 g
    elif num(d.get("salt_g")) is not None:                       # UK/EU-style: salt
        out["sodium_g"] = round(num(d.get("salt_g")) * factor / 2.5, 3)  # salt -> sodium
    return out                                                   # same format as the EasyOCR parser

reply = ask_vlm(img)                                             # the cookie panel again
print(reply)                                                     # raw reply
print(to_per100(reply))                                          # converted: should match OFF (24.14 fat, 68.97 carbs, ...)

In [ ]:
import pandas as pd                                              # tables
import time                                                      # pauses

off_path = "/kaggle/input/cleaned-off/off_filtered.csv"                     # from the Cleaned_OFF input (English OFF, same file as 10-ocr)
off = pd.read_csv(off_path, dtype={"code": str})                 # load; barcodes as text
fields = "image_ingredients_url,image_nutrition_url"             # the two photo links

def photo_links(code):                                           # barcode -> (ingredients link, nutrition link), with retry
    url = f"https://world.openfoodfacts.org/api/v2/product/{code}?fields={fields}"  # API address
    for attempt in range(2):                                     # try twice
        try:                                                     # safety net
            resp = requests.get(url, headers=headers, timeout=10)  # ask OFF
            if resp.status_code == 429:                          # "too many requests"
                time.sleep(10)                                   # wait, then retry
                continue                                         # next attempt
            product = resp.json().get("product", {})             # product part
            return product.get("image_ingredients_url"), product.get("image_nutrition_url")  # both links
        except Exception:                                        # network error
            time.sleep(2)                                        # short wait, retry
    return None, None                                            # failed twice

def add_links(table):                                            # add photo links to a table of products
    links = []                                                   # one pair per product
    for code in table["code"]:                                   # each barcode
        links.append(photo_links(code))                          # ask OFF
        time.sleep(1.0)                                          # be polite
    table = table.copy()                                         # don't change the original
    table["ingr_url"] = [a for a, b in links]                    # ingredients links
    table["nutr_url"] = [b for a, b in links]                    # nutrition links
    return table

picked = off.sample(160, random_state=42).reset_index(drop=True)  # SAME 160 development products as 10-ocr
test_pool = off[~off["code"].isin(picked["code"])]               # everything else
test_picked = test_pool.sample(160, random_state=7).reset_index(drop=True)  # SAME 160 test products as 10-ocr

picked = add_links(picked)                                       # development links (~3 min)
test_picked = add_links(test_picked)                             # test links (~3 min)
dev_nutr = picked[picked["nutr_url"].notna()].reset_index(drop=True)      # development products with a nutrition photo
test_nutr = test_picked[test_picked["nutr_url"].notna()].reset_index(drop=True)  # test products with a nutrition photo
print(len(dev_nutr), "development photos (36 before, more is fine)")  # count
print(len(test_nutr), "test photos (must be 151 to match the parser)")  # count: 151 = same test set

dev_nutr.to_csv("dev_nutr.csv", index=False)                     # save
test_nutr.to_csv("test_nutr.csv", index=False)                   # save

In [ ]:
print("development links done:", "nutr_url" in picked.columns)   # True = the first half finished before we stopped

In [ ]:
def add_links(table):                                            # add photo links, now WITH progress messages
    links = []                                                   # one pair per product
    start = time.time()                                          # clock
    for n, code in enumerate(table["code"]):                     # each barcode, n = 0, 1, 2, ...
        links.append(photo_links(code))                          # ask OFF
        time.sleep(1.0)                                          # be polite
        if (n + 1) % 20 == 0:                                    # every 20 products...
            print(n + 1, "of", len(table), "|", round(time.time() - start), "s")  # ...show progress and time
    table = table.copy()                                         # don't change the original
    table["ingr_url"] = [a for a, b in links]                    # ingredients links
    table["nutr_url"] = [b for a, b in links]                    # nutrition links
    return table

In [ ]:
  test_picked = add_links(test_picked)                             # test links, with progress

In [ ]:
   dev_nutr = picked[picked["nutr_url"].notna()].reset_index(drop=True)      # development products with a nutrition photo
   test_nutr = test_picked[test_picked["nutr_url"].notna()].reset_index(drop=True)  # test products with a nutrition photo
   print(len(dev_nutr), "development photos (36 before, more is fine)")      # count
   print(len(test_nutr), "test photos (must be 151 to match the parser)")    # count: 151 = same test set
   dev_nutr.to_csv("dev_nutr.csv", index=False)                              # save
   test_nutr.to_csv("test_nutr.csv", index=False)                            # save

In [ ]:
import pandas as pd                                              # tables
import time                                                      # timing


print(len(dev_nutr), "development photos")                       # should be 36

nutrients = ["protein_g", "fat_g", "carb_g", "sugars_g", "fiber_g", "sodium_g"]  # the six we score

def is_close(pred, true, name):                                  # SAME scoring rule as the parser
    floor = 0.05 if name == "sodium_g" else 0.5                  # smallest allowed error
    return abs(pred - true) <= max(0.2 * true, floor)            # within 20% or the floor

def fetch_full(url):                                             # photo link -> full-size picture
    r = requests.get(url.replace(".400.", ".full."), headers=headers, timeout=20)  # full resolution
    return Image.open(BytesIO(r.content)).convert("RGB")         # bytes -> picture

def score_vlm(table):                                            # run the VLM on a table of products and score it
    records, replies = [], {}                                    # scores; raw replies kept so we never have to rerun the model
    start = time.time()                                          # clock
    for i, prod in table.iterrows():                             # each product
        try:                                                     # safety net
            replies[i] = ask_vlm(fetch_full(prod["nutr_url"]))   # photo -> model reply
        except Exception as e:                                   # download / model problem
            replies[i] = {}                                      # nothing read
            print(i, "ERROR:", repr(e))                          # show it
        parsed = to_per100(replies[i])                           # our maths -> per 100 g
        for name in nutrients:                                   # each nutrient
            true, pred = prod[name], parsed.get(name)            # answer key, model value
            if pd.isna(true):                                    # OFF has no value
                continue                                         # skip (same rule)
            records.append({"i": i, "product_name": prod["product_name"], "nutrient": name, "true": true, "pred": pred,
                            "found": pred is not None, "correct": pred is not None and is_close(pred, true, name)})
        if i % 10 == 0:                                          # every 10 products...
            print(i, "done,", round(time.time() - start), "s")  # ...progress
    return pd.DataFrame(records), replies                        # scores + raw replies

vlm_dev, vlm_dev_replies = score_vlm(dev_nutr)                   # run on development
print("VLM dev found:", round(vlm_dev["found"].mean(), 2), "| correct:", round(vlm_dev["correct"].mean(), 2))  # vs parser v3: 0.58 / 0.29
print(vlm_dev.groupby("nutrient")[["found", "correct"]].mean().round(2))  # per nutrient

In [ ]:
miss = vlm_dev[(vlm_dev["nutrient"] == "sodium_g") & ~vlm_dev["found"]]  # sodium cases where nothing usable came back
print(len(miss), "sodium values not found")                      # how many
for i in miss["i"].head(12):                                     # first 12 of them
    r = vlm_dev_replies[i]                                       # the model's raw reply for that photo
    print(str(dev_nutr.loc[i, "product_name"])[:25], "| salt_g:", r.get("salt_g"),
          "| sodium:", r.get("sodium_amount"), r.get("sodium_unit"), "| basis:", r.get("basis"))  # what it said

In [ ]:
import matplotlib.pyplot as plt                                  # plotting

ids = list(miss["i"].head(12))                                   # the same 12 products as printed above
fig, axes = plt.subplots(3, 4, figsize=(24, 20))                 # 3 x 4 grid, big enough to read small print
for ax, i in zip(axes.flat, ids):                                # one slot per photo
    pic = fetch_full(dev_nutr.loc[i, "nutr_url"])                # full-size photo (same function the VLM used)
    ax.imshow(pic)                                               # draw it
    ax.set_title(str(dev_nutr.loc[i, "product_name"])[:30], fontsize=12)  # product name
    ax.axis("off")                                               # no axes
plt.tight_layout()                                               # tidy spacing
plt.show()                                                       # display


In [ ]:
prompt = ("Read the nutrition information panel in this photo. "
          "Return ONLY a JSON object with these keys: protein_g, fat_g, carb_g, sugars_g, fiber_g, "
          "salt_sodium_label, salt_sodium_amount, salt_sodium_unit, basis, serving_size_g. "
          "salt_sodium_label: the word used on the panel for the salt or sodium row (for example Salt, Sodium, Salz, Sel). "
          "salt_sodium_amount and salt_sodium_unit: the number and unit (g or mg) printed on that row. "
          "If there is a per 100 g (or per 100 ml) column, use it and set basis to \"per100\"; "
          "otherwise use the per serving values and set basis to \"per_serving\". "
          "Copy numbers exactly as printed, without doing any calculation."
          "If a value is printed as 'less than X' or '<X', return X. "
          "column_label: copy the heading printed above the numbers you used (for example 'per 100 g', 'Per 100g', "
          "'Amount per serving'). Use the per 100 g (or per 100 ml) column if there is one, otherwise the per serving column. "
          "Use null for anything not shown.")  # prompt v3
def grams_in(x):                                                 # serving size text -> grams ("2 cookies (29g)" -> 29.0)
    m = re.search(r"(\d+(?:\.\d+)?)\s*g", str(x))                # a number directly followed by g
    return float(m.group(1)) if m else num(x)                    # that number, or else the first number at all

def to_per100(d):                                                # reply -> per 100 g (v2 of our maths)
    if not d:                                                    # empty reply
        return {}
    factor = 1.0                                                 # per 100 g
    if "serv" in str(d.get("basis")).lower():                    # NEW: any spelling of "per serving" counts
        s = grams_in(d.get("serving_size_g"))                    # NEW: find the gram number, e.g. "2 cookies (29g)" -> 29
        if s:                                                    # NEW: only convert if a serving size was given...
            factor = 100 / s                                     # ...otherwise assume per 100 g instead of dropping everything
    out = {}                                                     # results
    for key in ["protein_g", "fat_g", "carb_g", "sugars_g", "fiber_g"]:  # gram nutrients
        v = num(d.get(key))                                      # number
        if v is not None and v * factor <= 100:                  # plausible
            out[key] = round(v * factor, 3)                      # per 100 g
    amount = num(d.get("salt_sodium_amount"))                    # NEW: one salt-or-sodium amount
    if amount is not None:                                       # given
        unit = str(d.get("salt_sodium_unit") or "g").lower()     # its unit
        grams = amount / 1000 if unit.startswith("mg") else amount  # mg -> g
        label = str(d.get("salt_sodium_label") or "salt").lower()   # its label word
        sodium = grams if label.startswith("sod") else grams / 2.5  # NEW: our code decides: sodium as is, salt (any language) / 2.5
        out["sodium_g"] = round(sodium * factor, 3)              # per 100 g
    return out

print(to_per100(ask_vlm(img)))                                   # quick check on the cookie panel again: should match OFF

In [ ]:
r = ask_vlm(img)                                                 # ask the model about the cookie panel
print("RAW:", r)                                                 # what the model wrote, before our maths
print("BASIS:", r.get("basis"), "| SERVING:", r.get("serving_size_g"))  # the two fields the conversion depends on

In [ ]:
img                                                              # the cookie panel the model is reading

In [ ]:
prompt = ("Read the nutrition information panel in this photo. "
          "Return ONLY a JSON object with these keys: protein_g, fat_g, carb_g, sugars_g, fiber_g, "
          "salt_sodium_label, salt_sodium_amount, salt_sodium_unit, column_label, serving_size_g. "
          "salt_sodium_label: the word used on the panel for the salt or sodium row (for example Salt, Sodium, Salz, Sel). "
          "salt_sodium_amount and salt_sodium_unit: the number and unit (g or mg) printed on that row. "
          "column_label: copy the heading printed above the numbers you used (for example 'per 100 g', 'Per 100g', "
          "'Amount per serving'). Use the per 100 g (or per 100 ml) column if there is one, otherwise the per serving column. "
          "Copy numbers exactly as printed, without doing any calculation. "
          "If a value is printed as 'less than X' or '<X', return X. "
          "Use null for anything not shown.")                     # prompt v3

def grams_in(x):                                                 # serving size text -> grams ("2 cookies (29g)" -> 29.0)
    m = re.search(r"(\d+(?:\.\d+)?)\s*g", str(x))                # a number directly followed by g
    return float(m.group(1)) if m else num(x)                    # that number, or else the first number at all

def to_per100(d):                                                # reply -> per 100 g
    if not d:                                                    # empty reply
        return {}
    factor = 1.0                                                 # per 100 g
    if not re.search(r"100\s?(g|ml)", str(d.get("column_label")).lower()):  # our code decides: no "100 g" in heading = per serving
        s = grams_in(d.get("serving_size_g"))                    # serving size in grams
        if s:                                                    # only convert if a serving size was found...
            factor = 100 / s                                     # ...otherwise assume per 100 g
    out = {}                                                     # results
    for key in ["protein_g", "fat_g", "carb_g", "sugars_g", "fiber_g"]:  # gram nutrients
        v = num(d.get(key))                                      # number
        if v is not None and v * factor <= 100:                  # plausible
            out[key] = round(v * factor, 3)                      # per 100 g
    amount = num(d.get("salt_sodium_amount"))                    # one salt-or-sodium amount
    if amount is not None:                                       # given
        unit = str(d.get("salt_sodium_unit") or "g").lower()     # its unit
        grams = amount / 1000 if unit.startswith("mg") else amount  # mg -> g
        label = str(d.get("salt_sodium_label") or "salt").lower()   # its label word
        sodium = grams if label.startswith("sod") else grams / 2.5  # sodium as is; salt (any language) / 2.5
        out["sodium_g"] = round(sodium * factor, 3)              # per 100 g
    return out

r = ask_vlm(img)                                                 # check on the cookie panel
print("RAW:", r)                                                 # what the model wrote
print("PER 100 g:", to_per100(r))                                # our conversion (should be fat 24.138, carbs 68.966, ...)

In [ ]:
vlm_dev3, vlm_dev3_replies = score_vlm(dev_nutr)                 # prompt v3 on the same 148 development photos
print("v2 dev correct: 0.55")                                    # earlier result, for comparison
print("v3 dev found:", round(vlm_dev3["found"].mean(), 2), "| correct:", round(vlm_dev3["correct"].mean(), 2))  # new result
print(vlm_dev3.groupby("nutrient")[["found", "correct"]].mean().round(2))  # per nutrient (watch sodium: was 0.32)

In [ ]:
wrong3 = vlm_dev3[vlm_dev3["found"] & ~vlm_dev3["correct"]]      # found a value, but outside the tolerance
ratio = (wrong3["pred"] / wrong3["true"]).replace([np.inf, -np.inf], np.nan)  # model value / OFF value (inf when OFF is 0)
print(len(wrong3), "wrong values")                               # how many
print(ratio.describe().round(2))                                 # typical ratios
print("about 1 (close but outside 20%):", ((ratio > 0.7) & (ratio < 1.4)).sum())   # near misses
print("serving mix-up (0.1-0.7 or 1.4-10):", (((ratio >= 0.1) & (ratio <= 0.7)) | ((ratio >= 1.4) & (ratio <= 10))).sum())  # wrong column or conversion?
print("OFF value is 0:", (wrong3["true"] == 0).sum())            # model gave a number where OFF says 0
print(wrong3.groupby("nutrient").size())                         # which nutrients

In [ ]:
mix = wrong3[(ratio >= 1.4) & (ratio <= 10)]                     # values 1.4-10x too big
for i in mix["i"].unique()[:12]:                                 # 12 products with that problem
    r = vlm_dev3_replies[i]                                      # the model's raw reply
    print(str(dev_nutr.loc[i, "product_name"])[:25], "| heading:", r.get("column_label"),
          "| serving:", r.get("serving_size_g"))                 # what heading and serving size it gave

In [ ]:
def grams_in_v4(x):                                              # serving size text -> grams, or None (no more guessing)
    m = re.search(r"(\d+(?:\.\d+)?)\s*g\b", str(x))              # a number directly followed by g ("29g", "100 g")
    return float(m.group(1)) if m else None                      # NEW: no fallback to "any number" (the McCain "7" bug)

def to_per100_v4(d):                                             # reply -> per 100 g (v4 of our maths)
    if not d:                                                    # empty reply
        return {}
    factor = 1.0                                                 # default: per 100 g
    heading = str(d.get("column_label")).lower()                 # the heading the model copied
    serving_style = any(w in heading for w in ["serv", "portion", "facts", "amount per"])  # NEW: clear per-serving signal
    if serving_style and not re.search(r"100\s?(g|ml)", heading):  # per serving AND no "100 g" in the heading
        s = grams_in_v4(d.get("serving_size_g"))                 # serving size in grams
        if s:                                                    # found
            factor = 100 / s                                     # per serving -> per 100 g
    out = {}                                                     # results
    for key in ["protein_g", "fat_g", "carb_g", "sugars_g", "fiber_g"]:  # gram nutrients
        v = num(d.get(key))                                      # number
        if v is not None and v * factor <= 100:                  # plausible
            out[key] = round(v * factor, 3)                      # per 100 g
    amount = num(d.get("salt_sodium_amount"))                    # salt-or-sodium amount
    if amount is not None:                                       # given
        unit = str(d.get("salt_sodium_unit") or "g").lower()     # unit
        grams = amount / 1000 if unit.startswith("mg") else amount  # mg -> g
        label = str(d.get("salt_sodium_label") or "salt").lower()   # label word
        sodium = grams if label.startswith("sod") else grams / 2.5  # sodium as is; salt / 2.5
        out["sodium_g"] = round(sodium * factor, 3)              # per 100 g
    return out

def rescore(replies, table, converter):                          # score SAVED replies with a conversion function (no model run)
    records = []                                                 # one entry per (product, nutrient)
    for i, prod in table.iterrows():                             # each product
        parsed = converter(replies.get(i, {}))                   # convert its saved reply
        for name in nutrients:                                   # each nutrient
            true, pred = prod[name], parsed.get(name)            # answer key, value
            if pd.isna(true):                                    # OFF has no value
                continue                                         # skip
            records.append({"i": i, "nutrient": name, "true": true, "pred": pred,
                            "found": pred is not None, "correct": pred is not None and is_close(pred, true, name)})
    return pd.DataFrame(records)

for label, conv in [("v3 maths", to_per100), ("v4 maths", to_per100_v4)]:  # same replies, two conversion versions
    res = rescore(vlm_dev3_replies, dev_nutr, conv)              # score
    print(label, "found:", round(res["found"].mean(), 2), "| correct:", round(res["correct"].mean(), 2))
vlm_dev4 = rescore(vlm_dev3_replies, dev_nutr, to_per100_v4)     # keep v4's details
print(vlm_dev4.groupby("nutrient")[["found", "correct"]].mean().round(2))  # per nutrient

In [ ]:
prompt_v3 = prompt                                               # keep v3's prompt safe (needed if v3 wins)

prompt = ("Read the nutrition information panel in this photo. "
          "Return ONLY a JSON object with these keys: protein_g, fat_g, carb_g, sugars_g, fiber_g, "
          "salt_sodium_label, salt_sodium_amount, salt_sodium_unit, per100_text, serving_size_g. "
          "salt_sodium_label: the word used on the panel for the salt or sodium row (for example Salt, Sodium, Salz, Sel). "
          "salt_sodium_amount and salt_sodium_unit: the number and unit (g or mg) printed on that row. "
          "per100_text: if the panel shows the text '100 g', '100g', '100 ml' or '100ml' anywhere "
          "(for example in a column heading 'per 100 g'), copy that text; otherwise null. "
          "serving_size_g: the serving size as printed (for example '29g'), or null. "
          "Use the per 100 g (or per 100 ml) column if there is one, otherwise the per serving column. "
          "Copy numbers exactly as printed, without doing any calculation. "
          "If a value is printed as 'less than X' or '<X', return X. "
          "Use null for anything not shown.")                     # prompt v5

def grams_in_v5(x):                                              # serving size -> grams, or None
    if isinstance(x, (int, float)):                              # NEW: a plain number (e.g. 29) is already grams
        return float(x)                                          # use it
    m = re.search(r"(\d+(?:\.\d+)?)\s*g\b", str(x))              # text with a unit, e.g. "29g", "30 g"
    if m:                                                        # found
        return float(m.group(1))                                 # that number
    if re.fullmatch(r"\s*\d+(?:\.\d+)?\s*", str(x)):             # NEW: text that is ONLY a number, e.g. "29"
        return float(x)                                          # also grams
    return None                                                  # anything else ("7 ovened servings") -> give up
def to_per100_v5(d):                                             # reply -> per 100 g (v5 maths)
    if not d:                                                    # empty reply
        return {}
    factor = 1.0                                                 # default: per 100 g
    has_100 = re.search(r"100\s?(g|ml)", str(d.get("per100_text")).lower()) is not None  # NEW: did the model SEE "100 g"?
    if not has_100:                                              # no per-100 column on the panel...
        s = grams_in_v5(d.get("serving_size_g"))                 # ...so use the serving size (strict: number + g)
        if s:                                                    # found
            factor = 100 / s                                     # per serving -> per 100 g
    out = {}                                                     # results
    for key in ["protein_g", "fat_g", "carb_g", "sugars_g", "fiber_g"]:  # gram nutrients
        v = num(d.get(key))                                      # number
        if v is not None and v * factor <= 100:                  # plausible
            out[key] = round(v * factor, 3)                      # per 100 g
    amount = num(d.get("salt_sodium_amount"))                    # salt-or-sodium amount
    if amount is not None:                                       # given
        unit = str(d.get("salt_sodium_unit") or "g").lower()     # unit
        grams = amount / 1000 if unit.startswith("mg") else amount  # mg -> g
        label = str(d.get("salt_sodium_label") or "salt").lower()   # label word
        sodium = grams if label.startswith("sod") else grams / 2.5  # sodium as is; salt / 2.5
        out["sodium_g"] = round(sodium * factor, 3)              # per 100 g
    return out

r = ask_vlm(img)                                                 # cookie check (US label: should have NO "100 g")
print("RAW:", r)                                                 # what the model wrote
print("PER 100 g:", to_per100_v5(r))                             # should be fat 24.138, carbs 68.966, ...

In [ ]:
print(grams_in_v5(29))                                           # should print 29.0
import inspect                                                   # tool to show a function's current code
print([line for line in inspect.getsource(to_per100_v5).splitlines() if "grams_in" in line])  # which grams_in does it call?

In [ ]:
_, vlm_dev5_replies = score_vlm(dev_nutr)                        # prompt v5 on the 148 development photos (~14 min)
vlm_dev5 = rescore(vlm_dev5_replies, dev_nutr, to_per100_v5)     # score with v5 maths
print("v3 (best so far) correct: 0.59")                          # for comparison
print("v5 found:", round(vlm_dev5["found"].mean(), 2), "| correct:", round(vlm_dev5["correct"].mean(), 2))  # result
print(vlm_dev5.groupby("nutrient")[["found", "correct"]].mean().round(2))  # per nutrient

In [ ]:
vlm_dev5_replies[125] = ask_vlm(fetch_full(dev_nutr.loc[125, "nutr_url"]))  # retry the photo that timed out
vlm_dev5 = rescore(vlm_dev5_replies, dev_nutr, to_per100_v5)     # rescore with the filled-in reply
print("v5 found:", round(vlm_dev5["found"].mean(), 2), "| correct:", round(vlm_dev5["correct"].mean(), 2))  # fair result
print(vlm_dev5.groupby("nutrient")[["found", "correct"]].mean().round(2))  # per nutrient

In [ ]:
import matplotlib.pyplot as plt                                  # plotting

prompt = prompt_v3                                               # FREEZE: v3 prompt is the one used for the test run

def show_page(p, per_page=12):                                   # show test photos number p*12 ... p*12+11
    idx = list(range(p * per_page, min((p + 1) * per_page, len(test_nutr))))  # which photos are on this page
    fig, axes = plt.subplots(3, 4, figsize=(24, 20))             # 3 x 4 grid, big enough to judge readability
    for ax, i in zip(axes.flat, idx):                            # one slot per photo
        try:                                                     # safety net
            ax.imshow(fetch_full(test_nutr.loc[i, "nutr_url"]))  # full-size photo
        except Exception:                                        # download problem
            ax.text(0.5, 0.5, "download failed", ha="center")    # say so in the slot
        ax.set_title(f"#{i}", fontsize=22)                       # just the number, no product name
        ax.axis("off")                                           # no axes
    for ax in axes.flat[len(idx):]:                              # empty slots on the last page
        ax.axis("off")                                           # hide them
    plt.tight_layout()                                           # tidy spacing
    plt.show()                                                   # display

show_page(0)                                                     # first page: photos 0-11

In [ ]:
show_page(1)

In [ ]:
show_page(2)

In [ ]:
show_page(3)

In [ ]:
show_page(4)

In [ ]:
show_page(5)

In [ ]:
show_page(6)

In [ ]:
show_page(7)

In [ ]:
show_page(8)

In [ ]:
show_page(9)

In [ ]:
show_page(10)

In [ ]:
show_page(11)

In [ ]:
show_page(12)

In [ ]:
hard = [0, 1, 2, 3, 4, 6, 8, 15, 18, 21, 25, 27, 28, 29, 30, 31, 32, 34, 37, 38, 42, 43, 44, 47,
        48, 49, 51, 53, 55, 56, 57, 58, 60, 62, 63, 65, 66, 67, 68, 70, 74, 76, 82, 83,
        84, 86, 87, 88, 91, 98, 99, 100, 101, 102, 104,
        108, 111, 112, 113, 115, 116, 118, 119, 121, 124, 125, 127, 130,
        132, 135, 137, 139, 140, 143, 144, 146, 147, 150]          # hard photos, labelled BEFORE the test run

test_nutr["photo_quality"] = ["hard" if i in hard else "clear" for i in range(len(test_nutr))]  # new column: clear/hard
test_nutr.to_csv("test_nutr_labelled.csv", index=False)          # save the labels
print(test_nutr["photo_quality"].value_counts())                 # expect hard 78, clear 73
print("frozen prompt is v3:", prompt == prompt_v3)               # must print True before the test run

In [ ]:
vlm_test, vlm_test_replies = score_vlm(test_nutr)                # FROZEN prompt v3 + v3 maths, on the 151 test photos
vlm_test["quality"] = vlm_test["i"].map(test_nutr["photo_quality"])  # attach clear/hard to every scored value
vlm_test.to_csv("vlm_nutrition_test.csv", index=False)           # save
print("TEST all   found:", round(vlm_test["found"].mean(), 2), "| correct:", round(vlm_test["correct"].mean(), 2))  # headline
print(vlm_test.groupby("quality")[["found", "correct"]].mean().round(2))  # clear vs hard photos
print(vlm_test.groupby("nutrient")[["found", "correct"]].mean().round(2))  # per nutrient

In [ ]:
import json                                                      # to save the raw model answers

dev_nutr.to_csv("dev_nutr.csv", index=False)                     # 148 development products (links + OFF answer key)
test_nutr.to_csv("test_nutr_labelled.csv", index=False)          # 151 test products + clear/hard label
vlm_dev3.to_csv("vlm_nutrition_dev_v3.csv", index=False)         # development scores, prompt v3 (frozen version)
vlm_dev5.to_csv("vlm_nutrition_dev_v5.csv", index=False)         # development scores, prompt v5 (for the version table)
vlm_test.to_csv("vlm_nutrition_test.csv", index=False)           # FINAL test scores (with clear/hard column)

replies = {"dev_v3": vlm_dev3_replies,                           # raw model answers, development, prompt v3
           "dev_v5": vlm_dev5_replies,                           # raw model answers, development, prompt v5
           "test_v3": vlm_test_replies}                          # raw model answers, test, prompt v3
with open("vlm_raw_replies.json", "w") as f:                     # open a file for writing
    json.dump({k: {str(i): r for i, r in v.items()} for k, v in replies.items()}, f)  # save (keys must be text in JSON)

with open("prompt_v3.txt", "w") as f:                            # the exact frozen prompt, for the report/appendix
    f.write(prompt_v3)                                           # write it
